In [ ]:
import pandas as pd
import yfinance as yf
from datetime import datetime
import plotly.express as px
import ast
import warnings
warnings.filterwarnings("ignore")



In [20]:

# start_date = df.index.min()
# end_date = df.index.max()

# macro_tickers = {
#     "Dollar": "CADUSD=X",   # US Dollar Index
#     "Oil": "CL=F",    # WTI Crude
#     "Bitcoin": "BTC-CAD",
#     "Gold": "GC=F"
# }

# macro_df = yf.download(
#     list(macro_tickers.values()),
#     start=start_date,
#     end=end_date,
#     auto_adjust=True,
#     progress=False
# )

# # Flatten columns (Close only)
# if isinstance(macro_df.columns, pd.MultiIndex):
#     macro_df = macro_df["Close"]

# # Rename to friendly names
# macro_df = macro_df.rename(columns={v: k for k, v in macro_tickers.items()})

# # Align with stock dates (inner join)
# combined = df.join(macro_df, how="inner")

# combined.head()
# # Scale macro series to 0-200
# macro_cols = ["Gold", "Bitcoin", "Oil", "Dollar"]

# scaled = combined.copy()
# for col in macro_cols:
#     series = scaled[col]
#     min_v = series.min()
#     max_v = series.max()
#     if max_v == min_v:
#         scaled[col] = 0  # or 100; constant series edge case
#     else:
#         scaled[col] = (series - min_v) / (max_v - min_v) * 200

# scaled.head()

In [21]:
def preprocess_tickers(ticker_list):
    tickers = [f"{t.strip().upper()}.TO" for t in ticker_list]

    return tickers


In [22]:
def parse_dividend_date(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value
    parsed = pd.to_datetime(value, errors="coerce")
    if pd.notna(parsed):
        return parsed.to_pydatetime()
    return None

In [23]:
def format_dividend_date(value):
    if value is None:
        return None
    return value.strftime("%d %b %Y")

In [24]:
def get_multi_metrics_data(df, dividend_thresh, apply_filters=True):

    results = []

    for _, row in df.iterrows():
        sector = row["sector"]
        tickers = ast.literal_eval(row["tickers"]) 
        for ticker in tickers:
            try:
                stock = yf.Ticker(ticker)
                info = stock.info

                industry = info.get("industry")
                dividend_pct = info.get("dividendYield")
                five_year_avg_dividend_yield = info.get("fiveYearAvgDividendYield")
                eps = info.get("trailingEps")
                forward_eps = info.get("forwardEps")
                beta = info.get("beta")
                peg_ratio = info.get("pegRatio")
                recommendation_key = info.get("recommendationKey")
                analyst_opinions = info.get("numberOfAnalystOpinions")
                target_high_price = info.get("targetHighPrice")
                current_price = info.get("currentPrice")
                target_low_price = info.get("targetLowPrice")
                all_time_high = info.get("allTimeHigh")
                all_time_low = info.get("allTimeLow")
                last_dividend_dt = parse_dividend_date(info.get("lastDividendDate"))
                ex_dividend_dt = parse_dividend_date(info.get("exDividendDate"))
                if last_dividend_dt is not None:
                    ex_dividend_dt = last_dividend_dt + pd.DateOffset(months=3)
                ex_dividend_date = format_dividend_date(ex_dividend_dt)
                last_dividend_date = format_dividend_date(last_dividend_dt)
                full_name = info.get("shortName")

                passes = (
                    dividend_pct is not None
                    and dividend_pct >= dividend_thresh
                    and eps is not None
                    and eps > 0
                )

                results.append({
                    "ticker": ticker,
                    "fullName": full_name,
                    "sector": sector,
                    "industry": industry,
                    "dividendYield": dividend_pct,
                    "fiveYearAvgDividendYield": five_year_avg_dividend_yield,
                    "trailingEps": eps,
                    "forwardEps": forward_eps,
                    "beta": beta,
                    "pegRatio": peg_ratio,
                    "recommendationKey": recommendation_key,
                    "numberOfAnalystOpinions": analyst_opinions,
                    "targetHighPrice": target_high_price,
                    "currentPrice": current_price,
                    "targetLowPrice": target_low_price,
                    "allTimeHigh": all_time_high,
                    "allTimeLow": all_time_low,
                    "exDividendDate": ex_dividend_date,
                    "lastDividendDate": last_dividend_date,
                    "passes_filters": passes
                })

                # avoid rate limiting
                # time.sleep(0.2)

            except Exception:
                results.append({
                    "ticker": ticker,
                    "fullName": full_name,
                    "sector": sector,
                    "industry": None,
                    "dividendYield": None,
                    "fiveYearAvgDividendYield": None,
                    "trailingEps": None,
                    "forwardEps": None,
                    "beta": None,
                    "pegRatio": None,
                    "recommendationKey": None,
                    "numberOfAnalystOpinions": None,
                    "targetHighPrice": None,
                    "currentPrice": None,
                    "targetLowPrice": None,
                    "allTimeHigh": None,
                    "allTimeLow": None,
                    "exDividendDate": None,
                    "lastDividendDate": None,
                    "passes_filters": False
                })

    df_out = pd.DataFrame(results)

    if apply_filters:
        df_out = df_out[df_out["passes_filters"]].copy()
    else:
        df_out = df_out.copy()

    # sort
    df_out = df_out.sort_values(
        ["sector", "dividendYield"],
        ascending=[True, False]
    ).reset_index(drop=True)

    return df_out

In [25]:
# dividend_df = get_multi_metrics_data(ticker_df, .50)
dividend_df = pd.read_csv("data/dividend_data.csv")
tickers_list = dividend_df["ticker"].tolist()

In [ ]:
def get_all_stock_data(
    tickers,
    period_days=2,
    day_period_years=10,
    threshold=5.0,
    top_k=15,
):
    if not tickers:
        return pd.DataFrame(), pd.DataFrame(), []

    # ── 1. Fetch minute-level data ────────────────────────────────────────────
    minute_data = yf.download(
        tickers,
        period=f"{period_days}d",
        interval="1m",
        progress=False,
        group_by="ticker",
        auto_adjust=False,
    )

    if minute_data.empty:
        return pd.DataFrame(), pd.DataFrame(), []

    # ── 2. Score each ticker by (highest - lowest) and filter by threshold ────
    range_scores = {}
    is_multi = isinstance(minute_data.columns, pd.MultiIndex)

    for ticker in tickers:
        # Get close prices with Adj Close fallback
        if is_multi:
            if ticker not in minute_data.columns.get_level_values(0):
                continue
            if not minute_data[ticker]["Close"].isna().all():
                close_prices = minute_data[ticker]["Close"]
            elif not minute_data[ticker]["Adj Close"].isna().all():
                close_prices = minute_data[ticker]["Adj Close"]
            else:
                continue
        else:
            # Single ticker — columns are flat
            if not minute_data["Close"].isna().all():
                close_prices = minute_data["Close"]
            elif not minute_data["Adj Close"].isna().all():
                close_prices = minute_data["Adj Close"]
            else:
                continue

        clean = close_prices.dropna()
        if clean.empty:
            continue

        price_range = clean.max() - clean.min()   # ← THE FIX: max - min, not std
        if price_range > threshold:
            range_scores[ticker] = price_range

    if not range_scores:
        return pd.DataFrame(), pd.DataFrame(), []

    # ── 3. Rank by range and pick top-k ──────────────────────────────────────
    ranked = sorted(range_scores.items(), key=lambda x: x[1], reverse=True)
    selected_tickers = [ticker for ticker, _ in ranked[:top_k]]

    # ── 4. Fetch day-level data for selected tickers ──────────────────────────
    day_data = yf.download(
        selected_tickers,
        period=f"{day_period_years}y",
        progress=False,
        group_by="ticker",
        auto_adjust=False,
    )

    if day_data.empty:
        return pd.DataFrame(), pd.DataFrame(), selected_tickers

    # ── 5. Build output DataFrames (Close with Adj Close fallback) ────────────
    def extract_close(data, tickers_list):
        """Slice close prices for each ticker, falling back to Adj Close."""
        is_multi = isinstance(data.columns, pd.MultiIndex)
        frames = {}
        for ticker in tickers_list:
            if is_multi:
                if not data[ticker]["Close"].isna().all():
                    frames[ticker] = data[ticker]["Close"]
                elif not data[ticker]["Adj Close"].isna().all():
                    frames[ticker] = data[ticker]["Adj Close"]
            else:
                if not data["Close"].isna().all():
                    frames[ticker] = data["Close"]
                elif not data["Adj Close"].isna().all():
                    frames[ticker] = data["Adj Close"]
        return pd.concat(frames, axis=1) if frames else pd.DataFrame()

    minute_df = extract_close(minute_data, selected_tickers)
    day_df    = extract_close(day_data,    selected_tickers)

    return minute_df, day_df, selected_tickers

In [27]:
# tickers
minute_df, day_df, ranked_tickers = get_all_stock_data(tickers_list)

In [12]:

# Load your saved stock data
# df = pd.read_csv("data/all_stock_data.csv", parse_dates=["Date"])
# df = df.set_index("Date").sort_index()
# df = df.fillna(0)
# Show ticker columns

In [22]:
def plot_series(data, names):
    # names: list like ["ABX.TO", "Gold"] or ["Gold", "Bitcoin", "Dollar"]
    missing = [n for n in names if n not in data.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    plot_df = data[names].reset_index()
    fig = px.line(
        plot_df,
        x="Date",
        y=names,
        title=" vs ".join(names),
        labels={"value": "Price", "variable": "Series"}
)
    fig.update_traces(visible="legendonly")
    fig.update_layout(width=800)
    fig.show()


In [14]:
# combined.to_csv("data/combined_data.csv", index=True, index_label="Date")
# dividend_df.to_csv("data/dividend_data.csv", index=False)

# combined = pd.read_csv("data/combined_data.csv", parse_dates=["Date"]).set_index("Date")

In [20]:

# Example usage
plot_series(minute_df, ranked_tickers)

### Python client

In [1]:
import json
import requests

FUNCTION_URL = "https://check-price-btbna6e3d5gpf5hs.eastus-01.azurewebsites.net/api/check-price"
# For local testing:
# FUNCTION_URL = "http://localhost:7071/api/check-price"

payload = {
    "token": "rifat1493",
    "products": [
        "Dyson or Shark Vacuum cleaner"

    ]
}

response = requests.post(
    FUNCTION_URL,
    json=payload,
    timeout=30
)

print("Status:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except ValueError:
    print(response.text)

Status: 200
{
  "results": [
    {
      "product": "Dyson or Shark Vacuum cleaner",
      "matches": [
        {
          "title": "70% Off Or More - Vacuum Cleaners & Floor Care",
          "link": "https://www.amazon.com/Vacuum-Cleaners-Floor-Care-Home-Kitchen/s?rh=n%3A510106%2Cp_8%3A70-",
          "percent": "70"
        },
        {
          "title": "The top Walmart deals this week include major savings on ...",
          "link": "https://shopping.yahoo.com/deals/article/the-top-walmart-deals-this-week-include-major-savings-on-dyson-hp-ninja-and-more-114803636.html",
          "percent": "70"
        },
        {
          "title": "Amazon's Big Spring Sale vacuum deals start right now",
          "link": "https://www.facebook.com/yahoolifestyle/posts/amazons-big-spring-sale-vacuum-deals-start-right-now-save-up-to-70-on-cordless-u/1301169815210277/",
          "percent": "70"
        },
        {
          "title": "DYSON VACUUMS ARE 70% OFF AT TARGET RIGHT ...",
          "li